In [1]:
#load variables
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
#create an API Client
from anthropic import Anthropic
client = Anthropic()
#change this according to: https://nextgen-beta.ica.ibm.com/ica/settings/global-models
model = "claude-haiku-4-5"

In [9]:
#multi-turn conversations
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages,system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
        "stop_sequences": stop_sequences
    }
    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message.content[0].text

In [37]:
import json

def generate_dataset():
    prompt = """
        Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex",
            "solution_criteria": "Key criteria for evaluating the solution"
            
        },
        ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [38]:
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:
        {test_case["task"]}
        
        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    
    return output

In [39]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        
        Original Task:
        <task>
            {test_case["task"]}
        </task>
        
        Solution to Evaluate:
        <solution>
            {output}
        </solution>
        
        Criteria you should use to evaluate the solution:
        <criteria>
            {test_case["solution_criteria"]}
        </criteria>
        
        Provide your evaluation as a structured JSON object with:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement  
        - "reasoning": A concise explanation of your assessment
        - "score": A number between 1-10
        
        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [30]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [33]:
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [34]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [40]:
from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"average score: {average_score}")
    return results

In [41]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

average score: 3.6666666666666665


In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Lambda S3 Event Processor Configuration\n\nHere's a comprehensive JSON configuration object for an AWS Lambda function that processes S3 events:\n\n```json\n{\n  \"FunctionName\": \"s3-event-processor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::123456789012:role/lambda-s3-execution-role\",\n  \"Handler\": \"index.lambda_handler\",\n  \"Timeout\": 60,\n  \"MemorySize\": 256,\n  \"Description\": \"Lambda function to process S3 bucket events\",\n  \"Environment\": {\n    \"Variables\": {\n      \"S3_BUCKET_NAME\": \"my-source-bucket\",\n      \"LOG_LEVEL\": \"INFO\",\n      \"ENVIRONMENT\": \"production\",\n      \"MAX_RETRIES\": \"3\"\n    }\n  },\n  \"VpcConfig\": {\n    \"SubnetIds\": [\n      \"subnet-12345678\"\n    ],\n    \"SecurityGroupIds\": [\n      \"sg-12345678\"\n    ]\n  },\n  \"Tags\": {\n    \"Project\": \"DataProcessing\",\n    \"Environment\": \"Production\",\n    \"ManagedBy\": \"Terraform\"\n  },\n  \"EphemeralStorage\": {\